In [2]:
import requests
import pandas as pd
import numpy as np
from scipy.stats import norm

# --- 常數定義 ---
DAYS_PER_YEAR = 365.25
SECONDS_PER_DAY = 24 * 3600


# --- Gate.io K 線資料抓取 ---
def get_gateio_kline(currency_pair: str, interval: str = "1s", limit: int = 720) -> pd.DataFrame:
    """
    從 Gate.io API 取得歷史 K 線資料
    文件: https://www.gate.io/docs/developers/apiv4/en/#market-k-line-chart
    """
    base_url = "https://api.gateio.ws/api/v4/spot/candlesticks"
    params = {
        "currency_pair": currency_pair.upper(),
        "interval": interval,
        "limit": limit,
    }

    response = requests.get(base_url, params=params, timeout=15)
    response.raise_for_status()
    data = response.json()

    # API 回傳格式: [[timestamp, volume_quote, close, high, low, open, volume_base, closed], ...]
    df = pd.DataFrame(
        data,
        columns=[
            "timestamp",
            "volume_quote",
            "close",
            "high",
            "low",
            "open",
            "volume_base",
            "closed",
        ],
    )

    df["timestamp"] = pd.to_datetime(df["timestamp"].astype(float), unit="s", utc=True)
    df[["open", "high", "low", "close"]] = df[["open", "high", "low", "close"]].astype(float)

    # 依時間排序（API 回傳通常是最新在前）
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df[["timestamp", "open", "high", "low", "close"]]


# --- Binance Futures K 線資料抓取 ---
def get_binance_futures_kline(symbol: str, interval: str = "1m", limit: int = 720) -> pd.DataFrame:
    """
    從 Binance USDT-M Futures API 取得歷史 K 線資料
    文件: https://binance-docs.github.io/apidocs/futures/en/#kline-candlestick-data
    """
    base_url = "https://fapi.binance.com/fapi/v1/klines"
    params = {
        "symbol": symbol.upper(),
        "interval": interval,
        "limit": limit,
    }

    response = requests.get(base_url, params=params, timeout=15)
    response.raise_for_status()
    data = response.json()

    if not isinstance(data, list) or len(data) == 0:
        raise ValueError(f"無法取得 {symbol} 的 K 線資料，請確認幣種與 interval 是否正確")

    # API 回傳格式:
    # [open_time, open, high, low, close, volume, close_time, quote_asset_volume,
    #  number_of_trades, taker_buy_base, taker_buy_quote, ignore]
    df = pd.DataFrame(
        data,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_asset_volume",
            "number_of_trades",
            "taker_buy_base",
            "taker_buy_quote",
            "ignore",
        ],
    )

    df["timestamp"] = pd.to_datetime(df["open_time"].astype(np.int64), unit="ms", utc=True)
    df[["open", "high", "low", "close"]] = df[["open", "high", "low", "close"]].astype(float)

    df = df.sort_values("timestamp").reset_index(drop=True)
    return df[["timestamp", "open", "high", "low", "close"]]


# --- 時間框換算（每個 interval 一天幾根 K） ---
def interval_to_bars_per_day(interval: str) -> float:
    mapping = {
        "1m": 1440,
        "3m": 480,
        "5m": 288,
        "15m": 96,
        "30m": 48,
        "1h": 24,
        "2h": 12,
        "4h": 6,
        "6h": 4,
        "8h": 3,
        "12h": 2,
        "1d": 1,
    }
    if interval not in mapping:
        raise ValueError(f"不支援的 interval: {interval}，請改用 {list(mapping.keys())}")
    return mapping[interval]


# --- 核心策略計算 ---
def calculate_optimal_market_making_params(
    asset: str,
    mid_price: float,
    daily_volatility_pct: float,
    target_order_fill_prob: float,
    order_refresh_time_sec: int,
    stop_loss_risk_prob: float,
    max_holding_time_days: float = 30.0,
    profit_factor: float = 2.0,
) -> dict:
    """
    根據 GBM 波動率模型，計算最優造市參數
    """
    if not (0 < target_order_fill_prob < 1):
        raise ValueError("target_order_fill_prob 必須在 (0, 1) 之間")
    if not (0 < stop_loss_risk_prob < 1):
        raise ValueError("stop_loss_risk_prob 必須在 (0, 1) 之間")

    daily_volatility = daily_volatility_pct / 100.0
    annual_volatility = daily_volatility * np.sqrt(DAYS_PER_YEAR)
    dt_order = order_refresh_time_sec / (DAYS_PER_YEAR * SECONDS_PER_DAY)
    dt_loss = max_holding_time_days / DAYS_PER_YEAR

    # 基礎掛單價差
    p_half_order = target_order_fill_prob / 2.0
    z_order = norm.ppf(p_half_order)
    base_spread_pct = (annual_volatility * np.sqrt(dt_order) * np.abs(z_order)) * 100

    # 止盈與止損
    profit_taking_spread_pct = base_spread_pct * profit_factor
    p_half_loss = stop_loss_risk_prob / 2.0
    z_loss = norm.ppf(p_half_loss)
    stop_loss_spread_pct = (annual_volatility * np.sqrt(dt_loss) * np.abs(z_loss)) * 100

    return {
        "asset": asset,
        "current_mid_price": mid_price,
        "order_refresh_time_sec": order_refresh_time_sec,
        "bid_spread": round(base_spread_pct, 4),
        "ask_spread": round(base_spread_pct, 4),
        "long_profit_taking_spread": round(profit_taking_spread_pct, 4),
        "short_profit_taking_spread": round(profit_taking_spread_pct, 4),
        "stop_loss_spread": round(stop_loss_spread_pct, 4),
        "z_score_order": round(np.abs(z_order), 4),
        "z_score_stop_loss": round(np.abs(z_loss), 4),
    }


def print_result(result: dict) -> None:
    print("\n--- 最優造市參數計算結果 ---")
    for key, value in result.items():
        if "_spread" in key or "time_sec" in key:
            unit = "%" if "spread" in key else "秒"
            print(f"{key:<30}: {value:>.4f} {unit}")
        elif key == "current_mid_price":
            print(f"{key:<30}: {value:>.4f} USDT")
        else:
            print(f"{key:<30}: {value}")


# --- 主流程：自動從 Binance Futures 取得波動率並計算策略 ---
def calculate_from_binance_futures(symbol: str = "XAUTUSDT", interval: str = "1m") -> dict:
    """
    從 Binance Futures 取得歷史資料，自動估算波動率並計算造市策略參數
    """
    df = get_binance_futures_kline(symbol=symbol, interval=interval, limit=720)

    # 計算對數報酬率
    log_returns = np.log(df["close"] / df["close"].shift(1)).dropna()
    if len(log_returns) < 30:
        raise ValueError("有效報酬率資料過少，請提高 limit 或改用較短 interval")

    bars_per_day = interval_to_bars_per_day(interval)
    period_vol = log_returns.std()
    daily_vol = period_vol * np.sqrt(bars_per_day)
    print(f"{symbol} 日化波動率估計值: {daily_vol*100:.2f}%")

    params = {
        "asset": symbol,
        "mid_price": float(df["close"].iloc[-1]),
        "daily_volatility_pct": daily_vol * 100,
        "target_order_fill_prob": 0.55,
        "order_refresh_time_sec": 3,
        "stop_loss_risk_prob": 0.06,
        "max_holding_time_days": 3/1440, # 3 分鐘換算成天
        "profit_factor": 1.5,
    }

    result = calculate_optimal_market_making_params(**params)
    print_result(result)
    return result


# --- 主執行 ---
if __name__ == "__main__":
    calculate_from_binance_futures("XAUTUSDT", interval="1m")

XAUTUSDT 日化波動率估計值: 2.72%

--- 最優造市參數計算結果 ---
asset                         : XAUTUSDT
current_mid_price             : 4611.1500 USDT
order_refresh_time_sec        : 3.0000 秒
bid_spread                    : 0.0096 %
ask_spread                    : 0.0096 %
long_profit_taking_spread     : 0.0144 %
short_profit_taking_spread    : 0.0144 %
stop_loss_spread              : 0.2338 %
z_score_order                 : 0.5978
z_score_stop_loss             : 1.8808
